# CASMI 2026 — closed-world Mode-A submission (offline)

**CLOSED-WORLD / CLASS-1 ONLY.** Candidates are the training structure library shipped in the
bundle. There is no external candidate expansion and no Mode B. This submission is an
**anchor**, not the final solution.

**Pipeline** (bundle `v2-A7`):

* The frozen V1 spectrum model is `V1_TL_1K_TESTSIM_STRICT`.
* The molecule aggregator is **MOST_CONFIDENT_SPECTRUM (A7)**. It was locked on MOL_DEV before
  HOST, and it uses the frozen calibration temperature shipped in `aggregation/`. RRF runs only if
  the bundle config names it and `EXPECTED_AGGREGATOR` is changed deliberately.

**Stages, in order:**

1. Locate the inputs.
2. Load the bundle.
3. Verify the manifest hashes.
4. Verify `CONFIG_HASH` and the aggregator/calibration contract.
5. Run the self-test at spectrum and molecule level.
6. Load the test data.
7. Parse adducts.
8. Retrieve candidates.
9. Compute the V1 features and scores.
10. Calibrate.
11. Aggregate with A7.
12. Deduplicate by connectivity.
13. Take the Top-25 representative SMILES.
14. Validate the submission.
15. Write `submission.csv`.
16. Write `run_report.json`.

Runs with internet **OFF**. Inputs are discovered under `/kaggle/input/` by schema, never by
hard-coded dataset names. For a local dry run, set `CASMI_KAGGLE_INPUT` and
`CASMI_KAGGLE_WORKING`. The notebook makes no network call and never submits anything.
Submitting is a manual Kaggle action.

In [ ]:
# CELL 1 -- environment: locate bundle / test / sample submission, verify every sha256 + CONFIG_HASH + the aggregator contract
import hashlib, json, os, sys, time
from pathlib import Path

T_START = time.time()
TIME_BUDGET_S = float(os.environ.get('CASMI_TIME_BUDGET_S', 8 * 3600))   # set below the competition limit
EMERGENCY_CAP_ENABLED = False      # default OFF: never degrade inference silently
EMERGENCY_CAP_N = 300              # closest-N candidates by abs ppm, only if enabled AND the projection exceeds the budget
EXPECTED_AGGREGATOR = 'MOST_CONFIDENT_SPECTRUM'   # the MOL_DEV-locked A7. Change ONLY to deliberately run an explicit baseline ('RRF')
INPUT_ROOT = Path(os.environ.get('CASMI_KAGGLE_INPUT', '/kaggle/input'))
WORK = Path(os.environ.get('CASMI_KAGGLE_WORKING', '/kaggle/working'))
WORK.mkdir(parents=True, exist_ok=True)

bundles = []
for p in INPUT_ROOT.rglob('manifest.json'):
    try:
        if json.loads(p.read_text(encoding='utf-8')).get('bundle_format', '').startswith('casmi-modeA-bundle'):
            bundles.append(p.parent)
    except Exception as e:
        print(f'skipping unreadable {p}: {type(e).__name__}')
if len(bundles) != 1:
    raise RuntimeError(f'expected exactly one CASMI bundle under {INPUT_ROOT}, found {bundles}')
BUNDLE_DIR = bundles[0]
sys.path.insert(0, str(BUNDLE_DIR / 'code'))

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
from casmi_infer.pipeline import TEST_REQUIRED, Bundle, RunLog, nearest_mass_ranking, resolve_neutral_mass, _rss_gb
from casmi_infer.validation import environment_report

tests = [p for p in INPUT_ROOT.rglob('*.parquet') if BUNDLE_DIR not in p.parents
         and set(TEST_REQUIRED) <= set(pq.read_schema(p).names)]
if len(tests) != 1:
    raise RuntimeError(f'expected exactly one test parquet with columns {TEST_REQUIRED}, found {tests}')
TEST_PATH = tests[0]
samples = [p for p in INPUT_ROOT.rglob('*.csv') if BUNDLE_DIR not in p.parents and list(pd.read_csv(p, nrows=0).columns) == ['molecule_id', 'smiles']]
if len(samples) != 1:
    raise RuntimeError(f'expected exactly one sample submission (molecule_id,smiles), found {samples}')
SAMPLE_PATH = samples[0]

log = RunLog()
t = time.time()
# raises on ANY sha256 / CONFIG_HASH mismatch, on a missing / mismatched frozen calibration, on a config model_id or
# feature order that differs from the shipped models, and on an aggregator config that names no aggregator
bundle = Bundle(BUNDLE_DIR, verify=True)
log.stage('load_bundle', time.time() - t)
# ANCHOR submission: the bundle must carry the FROZEN spectrum model (exported by 11_00 from outputs/v5/freeze/spectrum_model.json)
MODEL_INFO = json.loads((BUNDLE_DIR / 'models' / 'model_info.json').read_text(encoding='utf-8'))
REQUIRE_FROZEN_MODEL = True
if REQUIRE_FROZEN_MODEL and MODEL_INFO.get('freeze_status') != 'FROZEN':
    raise RuntimeError(f"bundle model {MODEL_INFO.get('model_id')} is not FROZEN ({MODEL_INFO.get('freeze_status')!r}); re-export the bundle with 11_00")
# the aggregator is whatever the LOCKED selection says -- never a silent RRF
if bundle.config.get('aggregator_name') != EXPECTED_AGGREGATOR:
    raise RuntimeError(f"bundle aggregator {bundle.config.get('aggregator_name')!r} != EXPECTED_AGGREGATOR {EXPECTED_AGGREGATOR!r}; "
                       're-export with 11_00 (PROMOTE_LOCKED_AGGREGATOR = True) or change EXPECTED_AGGREGATOR deliberately')
AGG_CONTRACT = bundle.aggregator_contract
BUNDLE_MANIFEST_SHA256 = hashlib.sha256((BUNDLE_DIR / 'manifest.json').read_bytes()).hexdigest()
SMILES_EXPORT_VALIDATION = (json.loads((BUNDLE_DIR / 'smiles_validation.json').read_text(encoding='utf-8'))
                            if (BUNDLE_DIR / 'smiles_validation.json').exists() else None)
ENV = environment_report()
print(json.dumps(ENV, indent=1))
print('bundle:', BUNDLE_DIR, '| version:', bundle.manifest.get('bundle_version'), '| CONFIG_HASH:', bundle.config['CONFIG_HASH'])
print('model:', bundle.ranker.model_id, '| folds:', len(bundle.ranker.boosters), '| feature order:', bundle.ranker.feature_names)
print('aggregator:', AGG_CONTRACT['name'], '| frozen T:', AGG_CONTRACT['temperature'], '| calibration sha256:', AGG_CONTRACT['calibration_sha256'],
      '| selection artifact sha256:', AGG_CONTRACT['selection_artifact_sha256'])
print('model freeze:', MODEL_INFO.get('freeze_status'), '| training protocol:', MODEL_INFO.get('training_protocol'),
      '| validation realism:', MODEL_INFO.get('validation_realism_status'), '| candidate universe:', bundle.config.get('candidate_universe_version'),
      '| manifest sha256:', BUNDLE_MANIFEST_SHA256[:16])
print('test:', TEST_PATH, '| sample:', SAMPLE_PATH, f'| rss {_rss_gb():.2f} GB')

In [ ]:
# CELL 1b -- bundle SELF-TEST before any hidden-test inference (fixture frozen inside the bundle; no HOST data needed)
# spectrum level: candidates, features, fold scores, fold-mean score, ranks; molecule level: calibrated probs, A7 selected
# spectrum + confidence, final Top-25 connectivity order. Normal backend first (numba if importable); numba parity failure
# -> numpy fallback (logged); numpy failure -> SelfTestFailed -> the notebook STOPS here and no submission can be written
from casmi_infer.backend import active_backend, numba_available
from casmi_infer.selftest import MISMATCH_KEYS, SelfTestFailed, run_selftest_with_fallback
from casmi_infer.validation import kaggle_offline_environment_proven
t = time.time()
SELFTEST = run_selftest_with_fallback(bundle)
log.stage('self_test', time.time() - t)
fin = SELFTEST['final']
print(f"self-test backend: initial={SELFTEST['self_test_backend_initial']} final={SELFTEST['self_test_backend_final']} "
      f"(numba available: {SELFTEST['numba_available']})")
print(f"fixture queries {fin['n_fixture_queries']} | molecules {fin['n_fixture_molecules']} | candidate rows {fin['n_candidate_rows']} | aggregator {fin['aggregator']}")
print('mismatches:', {k: fin[k] for k in MISMATCH_KEYS})
if SELFTEST['numba_fallback_used']:
    for line in ('!' * 72, 'NUMBA_PARITY_FAILED', 'USING_NUMPY_FALLBACK', '!' * 72):
        print(line)
print('SELF-TEST', 'PASS' if SELFTEST['passed'] else 'FAIL', '| active inference backend for the hidden test:', active_backend())
OFFLINE_PROVEN = kaggle_offline_environment_proven(input_root=INPUT_ROOT)

In [ ]:
# CELL 2 -- test load + schema validation + molecule grouping + unsupported adducts
t = time.time()
test = pd.read_parquet(TEST_PATH)
sample = pd.read_csv(SAMPLE_PATH)
missing = [c for c in TEST_REQUIRED if c not in test.columns]
assert not missing, f'test is missing {missing}'
assert not test['spectrum_id'].duplicated().any(), 'duplicate spectrum_id in test'
assert set(sample['molecule_id']) <= set(test['molecule_id']), 'sample molecules without test spectra'
test = test.reset_index(drop=True).assign(spectrum_order=lambda d: np.arange(len(d)))
spm = test.groupby('molecule_id').size()
print(f'{len(test)} spectra, {test.molecule_id.nunique()} molecules; spectra/molecule:', spm.describe()[['min', '50%', 'max']].to_dict())
UNSUPPORTED = sorted(a for a in test['adduct'].astype(str).unique() if not bundle.adducts.supported(a))
print('unsupported adducts (polarity-default fallback, logged per spectrum):', UNSUPPORTED)
log.stage('test_load', time.time() - t)

In [ ]:
# CELL 3 -- spectrum preprocessing: the training functions (remove_invalid_peaks + deterministic top-N)
from casmi_infer.spectrum import clean_query, query_meta, query_peak_hash
t = time.time()
Q = {}
for row in test.to_dict('records'):
    try:
        identity, sim = clean_query(row['ms2_mzs'], row['ms2_normalized_intensities'], row['precursor_mz'], bundle.sim_cfg['max_peaks_similarity'])
        Q[row['spectrum_id']] = {'row': row, 'sim': sim, 'hash': query_peak_hash(identity),
                                 'qmeta': query_meta(row['adduct'], row.get('ionization_mode'), row.get('instrument_type'), row.get('collision_energy_ev'))}
    except Exception as e:
        log.exceptions.append({'molecule_id': row['molecule_id'], 'spectrum_id': row['spectrum_id'], 'stage': 'preprocess',
                               'exception_class': type(e).__name__, 'message': str(e)[:300], 'fallback_type': 'nearest_mass'})
        Q[row['spectrum_id']] = {'row': row, 'failed': True}
log.stage('preprocess', time.time() - t)
print('preprocessed', len(Q), 'spectra; failures:', sum(1 for v in Q.values() if v.get('failed')))

In [ ]:
# CELL 4 -- candidate retrieval: neutral mass -> primary window -> 100 ppm -> 200 ppm -> nearest 25 (all logged)
t = time.time()
for sid, q in Q.items():
    row = q['row']
    try:
        nm, used = resolve_neutral_mass(bundle, row, log)
        q['neutral_mass'] = nm
        ci, ap, level = bundle.search.search(nm, bundle.primary_ppm(used), tuple(bundle.ppm['fallback_ppm']), bundle.ppm['nearest_n'])
        q.update(ci=ci, ap=ap, level=level)
        log.bump(f'retrieval_level_{level}')
        log.bump('candidates', len(ci))
        if level != 'primary':
            log.fallbacks.append({'molecule_id': row['molecule_id'], 'spectrum_id': sid, 'fallback_type': f'retrieval_{level}'})
    except Exception as e:
        log.exceptions.append({'molecule_id': row['molecule_id'], 'spectrum_id': sid, 'stage': 'retrieval', 'exception_class': type(e).__name__,
                               'message': str(e)[:300], 'fallback_type': 'nearest_mass'})
        q['failed'] = True
log.stage('retrieval', time.time() - t)
sizes = pd.Series({s: len(q.get('ci', [])) for s, q in Q.items()})
print('candidate pool size per spectrum:', sizes.describe()[['min', '50%', 'mean', 'max']].round(1).to_dict(), '| levels:',
      {k: v for k, v in log.counters.items() if k.startswith('retrieval_level')})

In [ ]:
# CELL 5 -- spectral evidence per molecule (chunked; never a global pair table) + runtime projection
from casmi_infer.features import candidate_features
t = time.time()
t_loop, n_done, cap_n = time.time(), 0, None
log.emergency_cap['enabled'] = EMERGENCY_CAP_ENABLED
for mid, g in test.groupby('molecule_id', sort=True):
    for sid in g['spectrum_id']:
        q = Q[sid]
        n_done += 1
        if q.get('failed'):
            continue
        try:
            ci, ap = q['ci'], q['ap']
            if cap_n is not None and len(ci) > cap_n:
                ci, ap = ci[:cap_n], ap[:cap_n]        # search() returns abs-ppm order -> the closest N
                log.emergency_cap['n_spectra_affected'] += 1
            q['feats'] = candidate_features(bundle.lib, ci, ap, q['sim'], q['qmeta'], q['hash'], bundle.sim_cfg, counters=log.counters)
        except Exception as e:
            log.exceptions.append({'molecule_id': mid, 'spectrum_id': sid, 'stage': 'spectral_evidence', 'exception_class': type(e).__name__,
                                   'message': str(e)[:300], 'fallback_type': 'nearest_mass'})
            q['failed'] = True
    log.tick_rss()
    elapsed = time.time() - T_START
    projected = elapsed + (time.time() - t_loop) / n_done * (len(test) - n_done)
    log.counters['projected_total_seconds'] = projected
    if projected > TIME_BUDGET_S and EMERGENCY_CAP_ENABLED and cap_n is None:
        cap_n = EMERGENCY_CAP_N
        log.emergency_cap.update(activated=True, n=cap_n, activated_after_spectra=n_done, projected_seconds_at_activation=projected)
        print(f'EMERGENCY CAP ACTIVATED after {n_done} spectra: closest {cap_n} candidates (projected {projected:.0f}s > budget {TIME_BUDGET_S:.0f}s)')
    if n_done % 100 < len(g):
        print(f'[evidence] {n_done}/{len(test)} spectra, {time.time() - t_loop:.0f}s, projected total {projected:.0f}s, rss {_rss_gb():.1f} GB')
log.stage('spectral_evidence', time.time() - t)

In [ ]:
# CELL 6 -- ranking: fold-mean LightGBM score, deterministic per-spectrum rank (score DESC, abs ppm ASC, key ASC)
from casmi_infer.ranker import rank_spectrum
t = time.time()
parts = []
for sid, q in Q.items():
    row = q['row']
    ranked, model_scored = None, False
    if not q.get('failed'):
        try:
            ranked = rank_spectrum(q['feats'], bundle.ranker.predict(q['feats']))
            model_scored = True
        except Exception as e:
            log.exceptions.append({'molecule_id': row['molecule_id'], 'spectrum_id': sid, 'stage': 'ranking', 'exception_class': type(e).__name__,
                                   'message': str(e)[:300], 'fallback_type': 'nearest_mass'})
    if ranked is None:   # the existing deterministic nearest-mass fallback for any failed spectrum (logged above)
        try:
            nm = q.get('neutral_mass') or resolve_neutral_mass(bundle, row, log)[0]
        except Exception:
            nm = None
        ranked = nearest_mass_ranking(bundle, nm, bundle.ppm['nearest_n'])
        log.bump('spectra_nearest_mass_fallback')
    parts.append(ranked.assign(molecule_id=row['molecule_id'], spectrum_id=sid, spectrum_order=row['spectrum_order'],
                               retrieval_level=q.get('level', 'fallback'), model_scored=model_scored))
per_spec = pd.concat(parts, ignore_index=True)
log.stage('ranking', time.time() - t)
print(f'{len(per_spec):,} ranked (spectrum, candidate) rows; model-scored spectra: {int(per_spec.groupby("spectrum_id")["model_scored"].first().sum())}/{len(Q)}')

In [ ]:
# CELL 7 -- calibrate (frozen T) -> aggregate with the CONFIGURED aggregator (A7 MOST_CONFIDENT_SPECTRUM) ->
#           connectivity dedup -> Top-25 -> representative SMILES
# A7: per molecule, the model-scored spectrum with the highest top-1 calibrated probability (ties: spectrum_id ASC) supplies
# the molecule ranking. Nearest-mass fallback spectra compete only when a molecule has no model-scored spectrum (counted).
from casmi_infer.pipeline import aggregate_molecules, selected_spectra
from casmi_infer.submission import pad_with_nearest
t = time.time()
mol_rank = aggregate_molecules(bundle, per_spec, counters=log.counters)
SELECTED = selected_spectra(bundle, per_spec)
if SELECTED is not None:
    print('A7 selected-spectrum confidence:', SELECTED['confidence'].describe()[['min', '50%', 'max']].round(4).to_dict())
    log.counters['a7_molecules_with_selected_spectrum'] = int(len(SELECTED))
TOP_K = int(bundle.config['top_k_submission'])
ranked_conn = {m: list(dict.fromkeys(g.sort_values('mol_rank', kind='mergesort')['conn_idx'].astype(int).tolist()))[:TOP_K]   # dedup BEFORE top-k
               for m, g in mol_rank.groupby('molecule_id')}
n_padded = 0
conn_final, smiles_by_mol = {}, {}
for mid in sample['molecule_id']:
    conn_list = ranked_conn.get(mid, [])
    if bundle.config.get('pad_to_top_k_with_nearest_mass') and len(conn_list) < TOP_K:   # the existing padding / zero-candidate policy
        g = per_spec[per_spec['molecule_id'] == mid]
        nm = pd.Series([Q[s].get('neutral_mass') for s in g['spectrum_id'].unique()], dtype=float).dropna()
        extra = bundle.search.nearest(float(nm.median()), TOP_K)[0].tolist() if len(nm) else []
        padded = pad_with_nearest(conn_list, extra, TOP_K)
        n_padded += int(len(padded) > len(conn_list))
        if not conn_list and padded:
            log.bump('molecules_zero_candidates_nearest_mass_fallback')
        conn_list = padded
    if not conn_list:
        log.exceptions.append({'molecule_id': mid, 'spectrum_id': None, 'stage': 'aggregation', 'exception_class': 'EmptyRanking',
                               'message': 'no candidate for any spectrum', 'fallback_type': 'none_available'})
    conn_final[mid] = conn_list
    smi = list(dict.fromkeys(bundle.smiles[c] for c in conn_list))   # order-preserving SMILES dedupe (distinct keys can share a representative SMILES)
    if len(smi) < len(conn_list):
        log.bump('duplicate_smiles_dropped', len(conn_list) - len(smi))
    smiles_by_mol[mid] = smi[:TOP_K]
log.counters['molecules_padded_with_nearest_mass'] = n_padded
log.stage('aggregation', time.time() - t)

In [ ]:
# CELL 8 -- submission validation (all must pass) + output; nothing is written after a failed self-test or validation
from casmi_infer.submission import build_submission, write_submission
from casmi_infer.validation import check_smiles_parse, validate_rankings, validate_submission
sub = build_submission(sample, smiles_by_molecule=smiles_by_mol)
problems = validate_submission(sub, sample, top_k=TOP_K)                 # columns, sample order/coverage, no empty/null, <= 25, no duplicate rows/SMILES
problems += validate_rankings(conn_final, list(sample['molecule_id']), top_k=TOP_K)   # every molecule once, non-empty, connectivity-unique
try:                                                                     # RDKit is optional here: parse if importable, else the export-time check
    from rdkit import Chem, RDLogger
    RDLogger.DisableLog('rdApp.*')
    PARSE_FN = lambda s: Chem.MolFromSmiles(s) is not None
except ImportError:
    PARSE_FN = None
emitted = [s for lst in smiles_by_mol.values() for s in lst]
SMILES_STATUS, smiles_problems = check_smiles_parse(emitted, SMILES_EXPORT_VALIDATION, PARSE_FN)
problems += smiles_problems
# deterministic ordering: rows follow the sample file; each ranking follows mol_rank (checked against the aggregator output)
order_bad = [m for m, c in ranked_conn.items() if conn_final.get(m, [])[:len(c)] != c]
problems += [f'{m}: output order differs from the aggregator ranking' for m in order_bad[:20]]
SUBMISSION_VALIDATION_STATUS = 'PASS' if not problems else 'FAIL'
print('submission validation:', SUBMISSION_VALIDATION_STATUS, '| SMILES parse check:', SMILES_STATUS, '| problems:', problems[:20] or 'none')
if problems:
    raise RuntimeError(f'{len(problems)} submission problems -- not writing submission.csv')
write_submission(sub, WORK / 'submission.csv', SELFTEST)     # refuses unless the self-test passed
print('wrote', WORK / 'submission.csv', sub.shape)

In [ ]:
# CELL 9 -- runtime report (every value derived from THIS run; completeness asserted)
from casmi_infer.validation import assert_run_report_complete
log.tick_rss()
FALLBACK_COUNTS = {**pd.Series([f['fallback_type'] for f in log.fallbacks], dtype=object).value_counts().astype(int).to_dict(),
                   **{k: int(v) for k, v in log.counters.items() if 'fallback' in k or k.startswith('retrieval_level_')},
                   'spectrum_exceptions': len(log.exceptions)}
report = {**log.to_dict(),
          # ---- v6.3 required fields ----
          'bundle_version': bundle.manifest.get('bundle_version'), 'CONFIG_HASH': bundle.config['CONFIG_HASH'], 'model_id': bundle.ranker.model_id,
          'aggregator': AGG_CONTRACT['name'], 'aggregator_config': bundle.config['aggregator'],
          'aggregator_artifact_hash': AGG_CONTRACT['selection_artifact_sha256'], 'calibration_hash': AGG_CONTRACT['calibration_sha256'],
          'calibration_temperature': AGG_CONTRACT['temperature'], 'candidate_universe_version': bundle.config.get('candidate_universe_version'),
          'self_test_status': 'PASS' if SELFTEST['passed'] else 'FAIL', 'backend': active_backend(), 'numba_available': bool(numba_available()),
          'numpy_fallback_used': bool(SELFTEST['numba_fallback_used']),
          'n_spectra': int(len(test)), 'n_molecules': int(test['molecule_id'].nunique()), 'candidate_pair_count': int(sizes.sum()),
          'fallback_counts': FALLBACK_COUNTS, 'runtime_seconds': time.time() - T_START, 'peak_ram': log.peak_rss_gb,
          'submission_rows': int(len(sub)), 'submission_validation_status': SUBMISSION_VALIDATION_STATUS, 'smiles_parse_check': SMILES_STATUS,
          # ---- existing fields (kept) ----
          'candidate_count_per_spectrum': sizes.describe().to_dict(), 'unsupported_adducts': UNSUPPORTED, 'environment': ENV,
          'time_budget_s': TIME_BUDGET_S, 'total_seconds': time.time() - T_START, 'mode': 'CLOSED-WORLD / CLASS-1 ONLY',
          'self_test_passed': bool(SELFTEST['passed']), 'self_test_backend_initial': SELFTEST['self_test_backend_initial'],
          'self_test_backend_final': SELFTEST['self_test_backend_final'], 'self_test_feature_mismatches': fin['feature_mismatches'],
          'self_test_score_mismatches': fin['score_mismatches'], 'self_test_rank_mismatches': fin['rank_mismatches'],
          'self_test_mismatches': {k: fin[k] for k in MISMATCH_KEYS}, 'self_test_attempts': SELFTEST['attempts'],
          'numba_fallback_used': bool(SELFTEST['numba_fallback_used']), 'inference_backend': active_backend(),
          'offline_environment_proven': bool(OFFLINE_PROVEN), 'peak_ram_gb': log.peak_rss_gb, 'candidate_count_total': int(sizes.sum()),
          'protocol': {'model_training_protocol': bundle.config.get('protocol'), 'model_training_protocol_hash': bundle.config.get('protocol_hash'),
                       'inference_eligibility': bundle.config.get('inference_eligibility')},
          'model_info': {k: v for k, v in MODEL_INFO.items() if k != 'source_dir'}, 'validation_realism_status': MODEL_INFO.get('validation_realism_status'),
          'bundle_manifest_sha256': BUNDLE_MANIFEST_SHA256, 'submission_sha256': hashlib.sha256((WORK / 'submission.csv').read_bytes()).hexdigest(),
          'deterministic': 'no randomness at inference: sorted molecule/spectrum loops, fold-mean LightGBM, fixed tie rules; submission_sha256 must repeat across runs'}
assert_run_report_complete(report)
(WORK / 'run_report.json').write_text(json.dumps(report, indent=2, default=str), encoding='utf-8')
print(json.dumps({k: report[k] for k in ('bundle_version', 'CONFIG_HASH', 'model_id', 'aggregator', 'calibration_temperature', 'self_test_status', 'backend',
                                         'numpy_fallback_used', 'n_spectra', 'n_molecules', 'fallback_counts', 'runtime_seconds', 'peak_ram',
                                         'submission_rows', 'submission_validation_status', 'submission_sha256')}, indent=1, default=str))
print('Submission is MANUAL. Afterwards log it locally with casmi.submissions.log_submission (public score typed in by hand).')
print('exceptions:', len(log.exceptions))